# VentureLens AI — Machine Learning Training Pipeline
**Real Dataset Citation**: manishkc06/startup-success-prediction (Kaggle)  
**Dataset URL**: https://www.kaggle.com/datasets/manishkc06/startup-success-prediction  
**Purpose**:
1. **Revenue/Cash-Flow Confidence Model**: RandomForestRegressor predicting operational cash-flow realization factor (0.45 - 1.0).
2. **Project Risk & Distress Classifier**: RandomForestClassifier predicting success probability (	ext{success})$ where $	ext{risk\_score} = 1 - P(	ext{success})$.
3. **Scenario Generator**: Explicit rule-based assumption generator (NOT ML) for optimistic (+20%), expected (confidence-adjusted), and pessimistic (-25%) cash flows.

> **Important Disclaimer**: ML predictions are estimates based on historical data, not guaranteed returns.


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
)
import joblib

DATA_PATH = os.path.join('..', 'data', 'startup_data.csv')
print('Data path exists:', os.path.exists(DATA_PATH))


## 1. Exploratory Data Analysis & Real Dataset Loading

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns')
print('Key columns:', ['labels', 'funding_rounds', 'milestones', 'relationships', 'funding_total_usd', 'age_last_funding_year'])
df[['labels', 'funding_rounds', 'milestones', 'relationships', 'age_last_funding_year', 'is_software', 'status']].head()


## 2. Feature Engineering & Target Derivation

In [ ]:
feature_cols = [
    'funding_rounds', 'milestones', 'relationships', 'age_last_funding_year',
    'avg_participants', 'is_software', 'is_enterprise', 'is_biotech', 'is_ecommerce'
]

# Impute missing values with median
for col in feature_cols:
    df[col] = df[col].fillna(df[col].median())

# Binary classification target: 1 = acquired / success, 0 = closed / distress
y_class = df['labels'].astype(int)

# Operational realization confidence score target (0.45 to 0.98)
milestone_factor = np.log1p(np.maximum(0, df['milestones'])) * 0.15
rel_factor = np.log1p(np.maximum(0, df['relationships'])) * 0.10
round_factor = np.clip(df['funding_rounds'] * 0.05, 0, 0.20)
raw_confidence = 0.50 + milestone_factor + rel_factor + round_factor
raw_confidence = np.where(y_class == 1, raw_confidence * 1.10, raw_confidence * 0.90)
y_reg = np.clip(raw_confidence / 1.35, 0.45, 0.98)

X = df[feature_cols].copy()
X_train, X_test, y_reg_train, y_reg_test, y_cls_train, y_cls_test = train_test_split(
    X, y_reg, y_class, test_size=0.20, random_state=42, stratify=y_class
)
print(f'Training samples: {len(X_train)}, Testing samples: {len(X_test)}')


## 3. Revenue Confidence Multiplier Model (RandomForestRegressor)

In [ ]:
reg_model = RandomForestRegressor(n_estimators=100, max_depth=6, min_samples_split=4, random_state=42)
reg_model.fit(X_train, y_reg_train)
y_reg_pred = reg_model.predict(X_test)

r2 = r2_score(y_reg_test, y_reg_pred)
mae = mean_absolute_error(y_reg_test, y_reg_pred)
rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))
print(f'Revenue Regressor Test Metrics -> R^2: {r2:.4f}, MAE: {mae:.4f}, RMSE: {rmse:.4f}')


## 4. Risk / Distress Classifier (RandomForestClassifier)

In [ ]:
cls_model = RandomForestClassifier(n_estimators=100, max_depth=5, min_samples_split=4, random_state=42)
cls_model.fit(X_train, y_cls_train)
y_cls_pred = cls_model.predict(X_test)
y_cls_proba = cls_model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_cls_test, y_cls_pred)
prec = precision_score(y_cls_test, y_cls_pred, zero_division=0)
rec = recall_score(y_cls_test, y_cls_pred, zero_division=0)
f1 = f1_score(y_cls_test, y_cls_pred, zero_division=0)
auc = roc_auc_score(y_cls_test, y_cls_proba)
print(f'Risk Classifier Test Metrics -> Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}, ROC-AUC: {auc:.4f}')


## 5. Deterministic Scenario Generator (Rule-based, NOT ML)
Explicitly computes optimistic (+20%), expected (confidence-adjusted), and pessimistic (-25%) cash flows.

In [ ]:
def generate_scenarios(base_cash_flows, confidence_multiplier):
    return {
        'expected': [round(cf * confidence_multiplier, 2) for cf in base_cash_flows],
        'optimistic': [round(cf * 1.20, 2) for cf in base_cash_flows],
        'pessimistic': [round(cf * 0.75, 2) for cf in base_cash_flows]
    }

# Example demonstration with spec project cash flows
example_cf = [300000, 300000, 300000, 300000, 300000]
scenarios = generate_scenarios(example_cf, confidence_multiplier=0.85)
print('Generated Scenarios:', scenarios)


## 6. Model Artifact Serialization

In [ ]:
os.makedirs('../backend/models', exist_ok=True)
joblib.dump(reg_model, '../backend/models/revenue_model.pkl')
joblib.dump(cls_model, '../backend/models/risk_model.pkl')

metrics = {
    'dataset_slug': 'manishkc06/startup-success-prediction',
    'revenue_model': {'r2': round(r2, 4), 'mae': round(mae, 4), 'rmse': round(rmse, 4)},
    'risk_classifier': {'accuracy': round(acc, 4), 'precision': round(prec, 4), 'recall': round(rec, 4), 'f1': round(f1, 4), 'roc_auc': round(auc, 4)}
}
with open('../backend/models/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Successfully exported models and metrics.')
